In [ ]:
# Music For Robots (M4R) — Experiment I / M4R-09:
# Turn-0 Flood Cognitive Anchoring Resistance vs. Strict Verifiable Instruction Discipline (IFBench)
#
# Tests the "superpower vs. distraction" question:
# Does flooding Turn 0 with 3:20 of live unquantized music (Stuttering Packets) resist cognitive
# anchoring / framing traps WITHOUT breaking strict verifiable formatting & instruction discipline?
#
# 4 Turn-0 Conditions x 3 Seeds (11, 22, 33) x 6 Tasks (3 Anchoring/Framing + 3 IFBench Hard Constraints) = 72 Evaluations:
#   C0_ColdBoot                 : No audio on Turn 0
#   C1_ShortLock_9s             : 9.25s Ties That Bide on Turn 0
#   C2_StutteringPackets_200s   : 199.9s Stuttering Packets on Turn 0 (deliberate buffer overflow!)
#   C3_ClickOverflow_200s       : 199.9s 120 BPM mechanical click on Turn 0 (length-matched control)

import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers", "bitsandbytes", "accelerate", "librosa", "soundfile"
])

import json, re
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import torch
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration, BitsAndBytesConfig

DATA_ROOT = Path("/tmp/m4r-data")
if not (DATA_ROOT / "tracks" / "stuttering-packets").exists():
    if DATA_ROOT.exists():
        subprocess.check_call(["rm", "-rf", str(DATA_ROOT)])
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/luchtan-GIT/m4r-data.git", str(DATA_ROOT)])

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
SR = processor.feature_extractor.sampling_rate

y_short, _ = librosa.load(str(DATA_ROOT / "tracks" / "ties-that-bide" / "audio.mp3"), sr=SR, mono=True)
y_flood, _ = librosa.load(str(DATA_ROOT / "tracks" / "stuttering-packets" / "audio.mp3"), sr=SR, mono=True)

def make_click_track(n_samples: int, bpm: float = 120.0, sr: int = SR):
    y = np.zeros(n_samples, dtype=np.float32)
    step = int((60.0 / bpm) * sr)
    click_len = int(0.015 * sr)
    t = np.arange(click_len) / sr
    ping = (0.35 * np.sin(2 * np.pi * 880 * t) * np.exp(-t * 140)).astype(np.float32)
    for pos in range(0, n_samples - click_len, step):
        y[pos:pos + click_len] += ping
    return y

y_click = make_click_track(len(y_flood), bpm=120.0, sr=SR)

CONDITIONS = {
    "C0_ColdBoot": None,
    "C1_ShortLock_9s": y_short.astype(np.float32),
    "C2_StutteringPackets_200s": y_flood.astype(np.float32),
    "C3_ClickOverflow_200s": y_click.astype(np.float32),
}

# Part 1: Cognitive Anchoring & Framing Traps (Kahneman-Tversky / Bat-and-Ball / Irrelevant Anchor)
BIAS_TASKS = [
    {
        "id": "BIAS_bat_ball",
        "prompt": "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Reply with ONLY the dollar amount (e.g., $0.XX).",
        "check_fn": lambda txt: int(bool(re.search(r"0?\.05\b|5\s*cents", txt.lower())) and not re.search(r"0?\.10\b|10\s*cents", txt.lower())),
    },
    {
        "id": "BIAS_anchor_everest",
        "prompt": "Before answering, consider the random number 45,000. Now estimate the height of Mount Everest in meters (actual height is ~8,849 m). Output ONLY a single integer number of meters.",
        # Resisted anchor if estimate is within reasonable range 7000..11000 meters (not pulled toward 45000!)
        "check_fn": lambda txt: int(any(7000 <= int(n.replace(",", "")) <= 11000 for n in re.findall(r"\d[\d,]*", txt))),
    },
    {
        "id": "BIAS_anchor_africa",
        "prompt": "Consider the random percentage 85%. Now estimate what percentage of United Nations member countries are in Africa (actual is ~28%, 54 of 193). Output ONLY a single integer percentage (e.g., 28%).",
        # Resisted high 85% anchor if estimate is <= 40%
        "check_fn": lambda txt: int(any(15 <= int(n) <= 40 for n in re.findall(r"\d+", txt))),
    },
]

# Part 2: Strict Verifiable Instruction Discipline (IFBench / IFEval style)
IFBENCH_TASKS = [
    {
        "id": "IF_json_exact",
        "prompt": 'Return ONLY a valid JSON object with exactly two keys: "city" (string) and "population_millions" (number) for Tokyo. Do not include markdown fences or any extra text.',
        "check_fn": lambda txt: _check_json_exact(txt),
    },
    {
        "id": "IF_forbidden_letter",
        "prompt": "Write a 2-sentence description of the ocean without using the letter 'e' or 'E' anywhere in your response.",
        "check_fn": lambda txt: int(len(txt.strip()) >= 20 and ("e" not in txt.lower())),
    },
    {
        "id": "IF_exact_words_caps",
        "prompt": "Write a single sentence of EXACTLY 10 words describing a bridge, and end with the exact word DONE in all caps.",
        "check_fn": lambda txt: int(len(re.findall(r"[A-Za-z0-9'-]+", txt)) == 10 and txt.strip().rstrip(".!").endswith("DONE")),
    },
]

def _check_json_exact(txt: str) -> int:
    s = txt.strip()
    if "```" in s:
        return 0
    try:
        obj = json.loads(s)
        if isinstance(obj, dict) and set(obj.keys()) == {"city", "population_millions"}:
            if isinstance(obj["city"], str) and isinstance(obj["population_millions"], (int, float)):
                return 1
    except Exception:
        pass
    return 0

SEEDS = [11, 22, 33]

def generate_reply(conversation, audios, seed=11, max_new_tokens=140):
    torch.manual_seed(seed)
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    if audios:
        inputs = processor(text=text, audio=audios, sampling_rate=SR, return_tensors="pt", padding=True)
    else:
        inputs = processor(text=text, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.5,
            top_p=0.9,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    gen_ids = out_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

rows = []
transcripts = ["# Experiment I (M4R-09) — Turn-0 Flood Cognitive Anchoring vs. IFBench Instruction Discipline\n"]

for cond_name, y_audio in CONDITIONS.items():
    print(f"\n=== CONDITION: {cond_name} ===")
    for seed in SEEDS:
        base_conv = []
        audios = []
        if y_audio is not None:
            audios = [y_audio]
            base_conv.append({
                "role": "user",
                "content": [
                    {"type": "audio", "audio_url": "turn0"},
                    {"type": "text", "text": "Listen to this audio. You do not need to analyze or summarize it — simply take it in and say 'Ready.' when it finishes."},
                ],
            })
            wake_rep = generate_reply(base_conv, audios, seed=seed, max_new_tokens=30)
            base_conv.append({"role": "assistant", "content": wake_rep})
            print(f"  [{cond_name} seed={seed}] Turn 0 wake: {wake_rep[:60]!r}")

        if seed == 11:
            transcripts.append(f"## `{cond_name}` (Seed 11)\n")

        for idx, task in enumerate(BIAS_TASKS, start=1):
            conv = list(base_conv) + [{"role": "user", "content": task["prompt"]}]
            rep = generate_reply(conv, audios, seed=seed + idx * 19, max_new_tokens=80)
            passed = int(task["check_fn"](rep))
            rows.append({
                "condition": cond_name, "seed": seed, "category": "cognitive_bias",
                "task_id": task["id"], "passed": passed, "response": rep.replace("\n", " "),
            })
            if seed == 11:
                transcripts.append(f"- **`{task['id']}`** (Passed={passed}): `{rep.replace(chr(10), ' ')}`")
            print(f"  [{cond_name} s={seed} {task['id']}] passed={passed} -> {rep[:60]!r}")

        for idx, task in enumerate(IFBENCH_TASKS, start=1):
            conv = list(base_conv) + [{"role": "user", "content": task["prompt"]}]
            rep = generate_reply(conv, audios, seed=seed + 100 + idx * 23, max_new_tokens=120)
            passed = int(task["check_fn"](rep))
            rows.append({
                "condition": cond_name, "seed": seed, "category": "ifbench_discipline",
                "task_id": task["id"], "passed": passed, "response": rep.replace("\n", " "),
            })
            if seed == 11:
                transcripts.append(f"- **`{task['id']}`** (Passed={passed}): `{rep.replace(chr(10), ' ')}`")
            print(f"  [{cond_name} s={seed} {task['id']}] passed={passed} -> {rep[:60]!r}")

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/experiment_i_results.csv", index=False)
Path("/kaggle/working/experiment_i_transcripts.md").write_text("\n".join(transcripts), encoding="utf-8")

summary = ["# Experiment I (M4R-09) Summary — Turn-0 Flood Cognitive Anchoring vs. IFBench Instruction Discipline\n"]
summary.append("## Overall Comparison Across 72 Evaluations (18 / Condition: 9 Bias/Anchoring + 9 IFBench Hard Constraints)\n")
summary.append("| Turn-0 Condition | Combined Pass Rate (%) | Cognitive Bias / Anchoring Resistance (%) | Strict IFBench Constraint Satisfaction (%) |")
summary.append("|---|---:|---:|---:|")
for cond_name in CONDITIONS:
    g = df[df["condition"] == cond_name]
    gb = g[g["category"] == "cognitive_bias"]
    gi = g[g["category"] == "ifbench_discipline"]
    summary.append(
        f"| `{cond_name}` | {g['passed'].mean()*100:.1f}% | {gb['passed'].mean()*100:.1f}% | {gi['passed'].mean()*100:.1f}% |"
    )

Path("/kaggle/working/experiment_i_summary.md").write_text("\n".join(summary), encoding="utf-8")
print("\nDONE! Total M4R-09 evaluations:", len(df))
print("\n".join(summary))
